# Crystals in Python, and ASE

A `Crystal` is the structure DISCUS holds, seen from Python: its cell, space group and atoms come
straight from DISCUS's arrays. The other way works too: build or change a structure with NumPy
or with [ASE](https://wiki.fysik.dtu.dk/ase/) (the Atomic Simulation Environment) and hand it to
DISCUS for the scattering calculation.

**Setup** (once, in the DiffuseCode folder, with your virtual environment active). ASE comes with
the package. Copy the examples folder and run the copy: Jupyter saves a notebook with its
results, which would change the file in your DiffuseCode clone and get in the way of `git pull`.

```bash
pip install ".[jupyter]"
cp -r python/examples ~/diffuse-examples
cd ~/diffuse-examples
jupyter lab
```

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

import diffuse as dc

## What a `Crystal` holds

`read_structure` reads a file into DISCUS (here a CIF: DISCUS applies the space group and builds
the supercell). The `Crystal` it returns knows the lattice, the space group and the number of
atoms.

In [ ]:
anatase = dc.read_structure("anatase.cif", supercell=(4, 4, 2))
print(anatase)
print("space group:", anatase.space_group)
print("lattice:    ", anatase.lattice)
print("cell volume:", round(anatase.volume, 3), "Å³")

`atoms()` gives the name and fractional coordinates (in units of the unit cell) of every atom,
`atom_types()` the atom types with their displacement parameter B and occupancy.
`cartesian()` gives positions in Å, `lattice_vectors()` the cell vectors as rows.

In [ ]:
names, xyz = anatase.atoms()
print(len(names), "atoms; the first three:")
for n, p in zip(names[:3], xyz[:3]):
    print(f"  {n:5s} {p}")

for name, b, occ in zip(*anatase.atom_types()):
    print(f"type {name:5s} B = {b:.2f} Å², occupancy {occ:.2f}")

print("lattice vectors (Å):")
print(anatase.lattice_vectors().round(4))
print("first atom in Å:", anatase.cartesian()[0])

`view()` shows the structure in 3D (rotate with the mouse; see `interactive_plots.ipynb` for the
options).

In [ ]:
anatase.view(style="ballstick", cell="unit")

## A crystal from NumPy arrays

`make_crystal` takes every atom: names, fractional coordinates and the lattice. Here a
copper crystal, 8 × 8 × 8 face-centred cubic cells, in which a random quarter of the atoms is
replaced by gold: a disordered Cu₃Au alloy.

In [ ]:
fcc = np.array([[0, 0, 0], [0.5, 0.5, 0], [0.5, 0, 0.5], [0, 0.5, 0.5]])
n = 8
cells = np.array([[i, j, k] for i in range(n) for j in range(n) for k in range(n)])
xyz = (cells[:, None, :] + fcc[None, :, :]).reshape(-1, 3)

rng = np.random.default_rng(42)
names = np.full(len(xyz), "CU")
names[rng.choice(len(xyz), size=len(xyz) // 4, replace=False)] = "AU"

alloy = dc.make_crystal((3.75, 3.75, 3.75, 90, 90, 90), names, xyz,
                        space_group="F m -3 m", b_iso={"CU": 0.6, "AU": 0.5}, cells=(n, n, n))
pure = dc.make_crystal((3.75, 3.75, 3.75, 90, 90, 90), ["CU"] * len(xyz), xyz,
                       space_group="F m -3 m", b_iso=0.6, cells=(n, n, n))
print(alloy)
print("types:", alloy.atom_types()[0], " atoms per type:",
      [int((alloy.atoms()[0] == t).sum()) for t in alloy.atom_types()[0]])

Both crystals stay available: a calculation on one sends it to DISCUS again if the other was
loaded since. With X-rays, gold scatters much more strongly than copper; on a random lattice the
pattern keeps the fcc peaks (no superstructure peaks: there is no order).

In [ ]:
kw = dict(qmin=1, qmax=8, dq=0.01, radiation="xray")
p_pure, p_alloy = dc.powder(pure, **kw), dc.powder(alloy, **kw)

fig, ax = plt.subplots(figsize=(8, 3.5))
ax.plot(p_pure.q, p_pure.intensity / p_pure.intensity.max(), label="Cu")
ax.plot(p_alloy.q, p_alloy.intensity / p_alloy.intensity.max(), label="random Cu$_3$Au", lw=1)
ax.set_xlabel("Q (Å$^{-1}$)")
ax.set_ylabel("intensity (normalised)")
ax.legend();

## Atom types

As in DISCUS, an atom type is a name **with** its B and occupancy: two oxygens with different B
are two types. B and occupancy can be one value, one per name (a dict) or one per atom.

In [ ]:
cr = dc.make_crystal((5, 5, 5, 90, 90, 90), ["TI", "O", "O"],
                     [[0, 0, 0], [0.5, 0, 0], [0, 0.5, 0]], b_iso=[0.3, 0.5, 0.9])
for name, b, occ in zip(*cr.atom_types()):
    print(f"{name:3s} B = {b}")

## ASE: from DISCUS to ASE

`to_ase()` turns the crystal into an `ase.Atoms` object: the cell is the whole crystal, positions
are in Å, elements come from the DISCUS names (which are kept in `atoms.arrays["discus_names"]`).
Everything ASE offers works on it, for example its neighbour list: the Ti–O bond lengths of
anatase.

In [ ]:
from ase.neighborlist import neighbor_list

atoms = anatase.to_ase()
print(atoms.get_chemical_formula(), atoms.cell.cellpar().round(3))

i, j, d = neighbor_list("ijd", atoms, cutoff=2.1)
symbols = np.array(atoms.get_chemical_symbols())
ti_o = d[(symbols[i] == "Ti") & (symbols[j] == "O")]
print("Ti-O bond lengths (Å):", np.unique(ti_o.round(3)))

ASE also writes many file formats (extended XYZ, VASP POSCAR, LAMMPS, ...). Here into a
temporary folder:

In [ ]:
import tempfile
from pathlib import Path

import ase.io

out = Path(tempfile.mkdtemp())
ase.io.write(out / "anatase.xyz", atoms)
ase.io.write(out / "POSCAR", atoms, format="vasp")
print(sorted(p.name for p in out.iterdir()))

## ASE: from ASE to DISCUS

`dc.from_ase()` hands any ASE structure to DISCUS. ASE has no space group, so the result is P1:
the ASE cell is one DISCUS unit cell holding every atom. A file that ASE reads, and DISCUS does
not, comes in the same way:

In [ ]:
import warnings

with warnings.catch_warnings():
    warnings.simplefilter("ignore")       # ASE notes how it reads the CIF's setting
    cif = ase.io.read("anatase.cif")
from_cif = dc.from_ase(cif, b_iso=1.0)
print(from_cif, from_cif.atom_types()[0])

### Nanoparticles

ASE builds clusters of many shapes. Two gold particles of about the same size, an icosahedron
and an octahedron: their powder patterns (Debye equation) differ, because the
icosahedron is not a piece of the fcc lattice.

In [ ]:
from ase.cluster import Icosahedron, Octahedron

shapes = {
    "icosahedron": Icosahedron("Au", noshells=5, latticeconstant=4.078),
    "octahedron": Octahedron("Au", length=7, cutoff=2, latticeconstant=4.078),
}
fig, ax = plt.subplots(figsize=(8, 3.5))
for label, particle in shapes.items():
    particle.center(vacuum=5.0)                     # a box around the particle
    crystal = dc.from_ase(particle, b_iso=0.6)
    p = dc.powder(crystal, qmin=1, qmax=10, dq=0.01, radiation="xray")
    ax.plot(p.q, p.intensity / p.intensity.max(), label=f"{label}, {crystal.natoms} atoms")
ax.set_xlabel("Q (Å$^{-1}$)")
ax.set_ylabel("intensity (normalised)")
ax.legend();

In [ ]:
dc.from_ase(shapes["icosahedron"]).view(cell=None)

## Displacement parameters

`adps()` gives every atom's displacement tensor: U11, U22, U33, U23, U13, U12 in Å² (the CIF
convention). An isotropic atom has the tensor of its B (U = B / 8π²). `make_crystal` takes
anisotropic tensors with `u_aniso`, per name or per atom; DISCUS then derives B (B = 8π² U<sub>eq</sub>).

In [ ]:
print("anatase, first atom (isotropic, B = 1):", anatase.adps()[0].round(5))

names, xyz = anatase.atoms()
flat_o = {"TI4+": (0.004, 0.004, 0.006, 0, 0, 0), "O2-": (0.003, 0.003, 0.012, 0, 0, 0)}
aniso = dc.make_crystal(anatase.lattice, names, xyz, u_aniso=flat_o, cells=anatase.supercell,
                        space_group=anatase.space_group)
for name, b, _occ in zip(*aniso.atom_types()):
    print(f"{name:5s} B = {b:.3f} Å² (from U)")

Atoms that vibrate more along c weaken reflections with a large l: compare the h0l planes.

In [ ]:
kw = dict(ll=(0, 0, 0), lr=(6, 0, 0), ul=(0, 0, 12), npoints=(121, 121))
iso_map = dc.fourier(anatase, **kw)
aniso_map = dc.fourier(aniso, **kw)
fig, axes = plt.subplots(1, 2, figsize=(9, 4), sharey=True)
for ax, m, title in zip(axes, (iso_map, aniso_map), ("isotropic, B = 1", "O vibrating along c")):
    m.plot(ax=ax, log=True)
    ax.set_title(title)
fig.tight_layout()

## Molecules

`molecules()` lists the molecules DISCUS holds; `make_crystal(molecules=...)` defines them, as
lists of atom indices. Here two water molecules of two types.

In [ ]:
names = ["O", "H", "H", "O", "H", "H"]
xyz = [[0.0, 0.0, 0.0], [0.096, 0.0, 0.0], [-0.024, 0.093, 0.0],
       [0.5, 0.5, 0.5], [0.596, 0.5, 0.5], [0.476, 0.593, 0.5]]
water = dc.make_crystal((10, 10, 10, 90, 90, 90), names, xyz, b_iso={"O": 1.0, "H": 2.0},
                        molecules=[[0, 1, 2], [3, 4, 5]], molecule_types=[1, 2])
types, members = water.molecules()
for t, m in zip(types, members):
    print(f"molecule type {t}: atoms {m.tolist()} = {[names[i] for i in m]}")

## Round trip

DISCUS → ASE → DISCUS keeps every atom; the Debye pattern, which depends only on the distances,
is the same.

In [ ]:
back = dc.from_ase(anatase.to_ase(), b_iso=dict(zip(*anatase.atom_types()[:2])))
p1 = dc.powder(anatase, qmin=1, qmax=6, dq=0.02)
p2 = dc.powder(back, qmin=1, qmax=6, dq=0.02)
print(back.natoms == anatase.natoms, np.allclose(p1.intensity, p2.intensity, rtol=1e-6))

In [ ]:
plt.close("all")